# Procesamiento y Clasificación de Ventas con Apache Spark

## Procesamiento de ventas mediante RDD

El objetivo de este desafío es utilizar Apache Spark para cargar,
limpiar, transformar y analizar un conjunto de datos de ventas
simuladas.

El procesamiento se realizará utilizando objetos RDD, aplicando
transformaciones y acciones para preparar los datos para una futura
etapa de análisis predictivo.

## 1. Configuración de Spark

Primero instalaremos PySpark y crearemos una sesión de Spark para
poder trabajar con RDD.

In [ ]:
# Instalamos PySpark para poder utilizar Apache Spark en Google Colab

!pip install -q pyspark openpyxl

In [ ]:
# Importamos las librerías necesarias

from pyspark.sql import SparkSession
from google.colab import files
import pandas as pd

# Se crea la sesión de Spark
spark = SparkSession.builder \
    .appName("Desafio_Spark_Ventas") \
    .getOrCreate()

# SparkContext permite trabajar directamente con RDD
sc = spark.sparkContext

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)

Spark iniciado correctamente
Versión de Spark: 4.0.4


## 2. Carga de los datos

El archivo entregado se encuentra en formato Excel. Lo cargaremos
primero en Colab y posteriormente lo convertiremos a CSV para que
Spark pueda leerlo mediante textFile().

In [ ]:
# Se sube el archivo desde el computador
archivo = files.upload()

# Se obtiene el nombre del archivo cargado
nombre_archivo = list(archivo.keys())[0]

print("Archivo cargado:", nombre_archivo)

Saving 07. Apoyo desafío - ventas_simuladas.xlsx to 07. Apoyo desafío - ventas_simuladas (1).xlsx
Archivo cargado: 07. Apoyo desafío - ventas_simuladas (1).xlsx


In [ ]:
# Se lee la hoja que contiene los datos de ventas

df = pd.read_excel(
    nombre_archivo,
    sheet_name="ventas_simuladas.csv",
    dtype=str
)

# Elimina espacios innecesarios de los nombres de columnas
df.columns = df.columns.str.strip()

print("Cantidad de registros:", len(df))
print("Columnas:")
print(list(df.columns))

Cantidad de registros: 200
Columnas:
['Sucursal', 'Producto', 'Cantidad', 'Precio_Unitario', 'Monto_Total', 'Fecha_Hora']


## 3. Creación del RDD base

Convertiremos el archivo a CSV y luego utilizaremos textFile()
para crear el RDD base.

En este punto cada elemento del RDD corresponde a una línea del
archivo.

In [ ]:
# Se guarda el archivo como csv, este será el archivo que se leera con Spark

df.to_csv(
    "ventas_simuladas.csv",
    index=False,
    encoding="utf-8"
)

# Creamos el RDD base leyendo el archivo CSV
rdd = sc.textFile("ventas_simuladas.csv")

# count() es una acción que cuenta los elementos del RDD
print("Cantidad de líneas:", rdd.count())

Cantidad de líneas: 201


### Exploración del RDD

Utilizaremos take() para visualizar las primeras líneas del RDD
y comprobar que los datos fueron cargados correctamente.

In [ ]:
# take() permite obtener una cantidad determinada de elementos del RDD

for linea in rdd.take(6):
    print(linea)

Sucursal,Producto,Cantidad,Precio_Unitario,Monto_Total,Fecha_Hora
Sucursal A,Pan,3,989.78,2969.34,2023-01-04 17:17:12
Sucursal A,Aceite,4,563.57,2254.28,2023-01-03 14:34:53
Sucursal B,Galletas,5,1339.04,6695-02-01 00:00:00,invalid-date
Sucursal B,Galletas,4,1180.5,4722.0,2023-01-05 02:34:24
Sucursal D,Pan,3,2194.99,6584.97,2023-01-07 18:05:40


## 4. Limpieza y estructuración de los datos

Ahora comenzaremos a transformar el RDD.

Primero eliminaremos el encabezado y luego separaremos cada línea
en sus diferentes campos.

In [ ]:
# Se guarda la primera línea, que corresponde al encabezado
encabezado = rdd.first()

# Elimin el encabezado del RDD
rdd_datos = rdd.filter(lambda x: x != encabezado)

print("Registros sin encabezado:", rdd_datos.count())

Registros sin encabezado: 200


In [ ]:
# map() aplica una función a cada elemento del RDD y split(",") separa cada línea utilizando la coma

rdd_separado = rdd_datos.map(
    lambda x: x.split(",")
)

# mostrar algunos registros para comprobar la transformación
for registro in rdd_separado.take(5):
    print(registro)

['Sucursal A', 'Pan', '3', '989.78', '2969.34', '2023-01-04 17:17:12']
['Sucursal A', 'Aceite', '4', '563.57', '2254.28', '2023-01-03 14:34:53']
['Sucursal B', 'Galletas', '5', '1339.04', '6695-02-01 00:00:00', 'invalid-date']
['Sucursal B', 'Galletas', '4', '1180.5', '4722.0', '2023-01-05 02:34:24']
['Sucursal D', 'Pan', '3', '2194.99', '6584.97', '2023-01-07 18:05:40']


### Filtrado de registros incompletos

El archivo contiene algunos registros con campos vacíos.

Utilizaremos filter() para conservar solamente aquellos registros
que tengan sus seis campos completos.

In [ ]:
# Se conservan los registros que tengan exactamente 6 campos y que no tengan campos vacíos

rdd_validos = rdd_separado.filter(
    lambda x: len(x) == 6 and all(campo.strip() != "" for campo in x)
)

print("Registros después del filtrado:", rdd_validos.count())

Registros después del filtrado: 182


### Conversión de los tipos de datos

Las columnas numéricas deben convertirse a los tipos correspondientes:

- Cantidad → entero (int)
- Precio_Unitario → decimal (float)
- Monto_Total → decimal (float)

Además, se validará la fecha y hora.

Como existen registros con errores, utilizaremos try/except para
evitar que el procesamiento se detenga.

In [ ]:
from datetime import datetime

# Función para convertir y validar cada registro

def limpiar_registro(x):
    try:
        # Convertirros campos numéricos
        cantidad = int(float(x[2]))
        precio = float(x[3])
        monto = float(x[4])

        # Convertir la fecha y hora
        fecha_hora = datetime.strptime(
            x[5],
            "%Y-%m-%d %H:%M:%S"
        )

        # Registro limpio
        return (
            x[0],          # Sucursal
            x[1],          # Producto
            cantidad,      # Cantidad
            precio,        # Precio unitario
            monto,         # Monto total
            fecha_hora     # Fecha y hora
        )

    except:
        # Si existe algún error, se descarta el registro
        return None


# Se aplica la función a todos los registros
rdd_limpio = rdd_validos.map(limpiar_registro)

# Se elimina los registros que no pudieron ser convertidos
rdd_limpio = rdd_limpio.filter(lambda x: x is not None)

print("Registros después de la limpieza:", rdd_limpio.count())

Registros después de la limpieza: 132


### Revisión del RDD limpio

A continuación revisaremos algunos registros para comprobar que
los datos numéricos y temporales fueron correctamente transformados.

In [ ]:
# Mostramos los primeros registros después de la limpieza

for registro in rdd_limpio.take(5):
    print(registro)

('Sucursal A', 'Pan', 3, 989.78, 2969.34, datetime.datetime(2023, 1, 4, 17, 17, 12))
('Sucursal A', 'Aceite', 4, 563.57, 2254.28, datetime.datetime(2023, 1, 3, 14, 34, 53))
('Sucursal B', 'Galletas', 4, 1180.5, 4722.0, datetime.datetime(2023, 1, 5, 2, 34, 24))
('Sucursal D', 'Pan', 3, 2194.99, 6584.97, datetime.datetime(2023, 1, 7, 18, 5, 40))
('Sucursal B', 'Atún', 3, 2470.44, 7411.32, datetime.datetime(2023, 1, 6, 23, 36, 37))


### Separación de fecha y hora

Para facilitar el análisis temporal, separaremos la variable
`Fecha_Hora` en dos variables: Fecha y Hora.

In [ ]:
# Separamos la fecha y la hora utilizando strftime()

rdd_final = rdd_limpio.map(
    lambda x: (
        x[0],                         # Sucursal
        x[1],                         # Producto
        x[2],                         # Cantidad
        x[3],                         # Precio unitario
        x[4],                         # Monto total
        x[5].strftime("%Y-%m-%d"),    # Fecha
        x[5].strftime("%H:%M:%S")     # Hora
    )
)

# Revisamos el resultado
for registro in rdd_final.take(5):
    print(registro)

('Sucursal A', 'Pan', 3, 989.78, 2969.34, '2023-01-04', '17:17:12')
('Sucursal A', 'Aceite', 4, 563.57, 2254.28, '2023-01-03', '14:34:53')
('Sucursal B', 'Galletas', 4, 1180.5, 4722.0, '2023-01-05', '02:34:24')
('Sucursal D', 'Pan', 3, 2194.99, 6584.97, '2023-01-07', '18:05:40')
('Sucursal B', 'Atún', 3, 2470.44, 7411.32, '2023-01-06', '23:36:37')


## 5. Análisis del RDD limpio

Una vez realizada la limpieza, utilizaremos acciones y
transformaciones de Spark para obtener indicadores sobre el
comportamiento de las ventas.

Se calcularán tres indicadores principales:

1. Total de ventas por sucursal.
2. Monto promedio por compra.
3. Cantidad de transacciones por franja horaria.

### 5.1. Total de ventas por sucursal

Para obtener este indicador crearemos pares
(Sucursal, Monto_Total) y utilizaremos reduceByKey() para sumar
las ventas correspondientes a cada sucursal.

In [ ]:
# Se crean pares Sucursal - Monto_Total

ventas_sucursal = rdd_final.map(
    lambda x: (x[0], x[4])
)

# Suma de los montos de cada sucursal
ventas_sucursal = ventas_sucursal.reduceByKey(
    lambda a, b: a + b
)

# collect() devuelve los resultados para poder visualizarlos
resultado = ventas_sucursal.collect()

print("Ventas totales por sucursal:\n")

for sucursal, total in resultado:
    print(f"{sucursal}: ${total:,.2f}")

Ventas totales por sucursal:

Sucursal A: $119,436.26
Sucursal B: $149,711.39
Sucursal D: $159,766.71
Sucursal C: $146,544.27


### 5.2. Monto promedio por compra

Para obtener el promedio necesitamos conocer dos valores:

- La suma de los montos.
- La cantidad de transacciones.

Utilizaremos aggregateByKey() para obtener ambos valores por
sucursal.

In [ ]:
# Se crean pares Sucursal - Monto_Total
ventas = rdd_final.map(
    lambda x: (x[0], x[4])
)

# aggregateByKey() acumularásuma de ventas y cantidad de compras
promedios = ventas.aggregateByKey(
    (0.0, 0),                         # valor inicial
    lambda acc, valor: (
        acc[0] + valor,               # suma
        acc[1] + 1                    # cantidad
    ),
    lambda acc1, acc2: (
        acc1[0] + acc2[0],            # combina las sumas
        acc1[1] + acc2[1]             # combina las cantidades
    )
)

# Promedio
promedios = promedios.map(
    lambda x: (x[0], x[1][0] / x[1][1])
)

# resultados
resultado_promedio = promedios.collect()

print("Monto promedio por compra:\n")

for sucursal, promedio in resultado_promedio:
    print(f"{sucursal}: ${promedio:,.2f}")

Monto promedio por compra:

Sucursal A: $3,981.21
Sucursal B: $4,536.71
Sucursal D: $4,318.02
Sucursal C: $4,579.51


### 5.3. Transacciones por franja horaria

Para analizar el comportamiento temporal de las ventas,
clasificaremos las transacciones en cuatro franjas:

- Madrugada: 00:00–05:59
- Mañana: 06:00–11:59
- Tarde: 12:00–17:59
- Noche: 18:00–23:59

In [ ]:
# Función para determinar la franja horaria
def franja_horaria(hora):
    h = int(hora.split(":")[0])
    if h < 6:
        return "Madrugada"
    elif h < 12:
        return "Mañana"
    elif h < 18:
        return "Tarde"
    else:
        return "Noche"


# Se crean pares Franja - 1. El 1 representa una transacción
transacciones = rdd_final.map(
    lambda x: (franja_horaria(x[6]), 1)
)

# Se suman las transacciones de cada franja
transacciones = transacciones.reduceByKey(
    lambda a, b: a + b
)

# Resultados
resultado_franjas = transacciones.collect()

print("Transacciones por franja horaria:\n")

for franja, cantidad in resultado_franjas:
    print(f"{franja}: {cantidad}")

Transacciones por franja horaria:

Tarde: 39
Madrugada: 28
Noche: 33
Mañana: 32


# Prueba -Modelo predictivo en Apache Spark MLlib_

## 1. Preparación del dataset para MLlib

A partir del RDD limpio obtenido en el desafío anterior se construirá un DataFrame de Spark para preparar los datos para el modelo predictivo.

### 1.1 Conversion del RDD limpio a dataframe

A partir del RDD limpio obtenido en el desafío anterior, se construirá un DataFrame de Spark para preparar los datos para el modelo predictivo.

In [ ]:
### Conversión del RDD limpio a Dataframe

# Defir los nombres de las columnas
columnas = [
    "Sucursal",
    "Producto",
    "Cantidad",
    "Precio_Unitario",
    "Monto_Total",
    "Fecha",
    "Hora"
]

# Convertir el RDD limpio en un DataFrame de Spark
df_ml = spark.createDataFrame(
    rdd_final,
    columnas
)

# Revisar la estructura
df_ml.printSchema()

# Mostrar algunos registros
df_ml.show(5)

root
 |-- Sucursal: string (nullable = true)
 |-- Producto: string (nullable = true)
 |-- Cantidad: long (nullable = true)
 |-- Precio_Unitario: double (nullable = true)
 |-- Monto_Total: double (nullable = true)
 |-- Fecha: string (nullable = true)
 |-- Hora: string (nullable = true)

+----------+--------+--------+---------------+-----------+----------+--------+
|  Sucursal|Producto|Cantidad|Precio_Unitario|Monto_Total|     Fecha|    Hora|
+----------+--------+--------+---------------+-----------+----------+--------+
|Sucursal A|     Pan|       3|         989.78|    2969.34|2023-01-04|17:17:12|
|Sucursal A|  Aceite|       4|         563.57|    2254.28|2023-01-03|14:34:53|
|Sucursal B|Galletas|       4|         1180.5|     4722.0|2023-01-05|02:34:24|
|Sucursal D|     Pan|       3|        2194.99|    6584.97|2023-01-07|18:05:40|
|Sucursal B|    Atún|       3|        2470.44|    7411.32|2023-01-06|23:36:37|
+----------+--------+--------+---------------+-----------+----------+--------+
on

### 1.2 Creación de la variable objetivo

La variable label corresponde a la variable que el modelo deberá predecir.

Se define:
- label = 1: transacción riesgosa.
- label = 0: transacción normal.

La condición utilizada será la mencionada como sugerencia en la prueba:
- Monto total mayor a $7.000.
- Venta realizada durante la madrugada (00:00 a 05:59).

In [ ]:
# Creación de la variable label

# Extracción hora desde el campo Hora
df_ml = df_ml.withColumn(
    "Hora_num",
    F.hour(F.to_timestamp("Hora", "HH:mm:ss"))
)

# Creación la variable objetivo
# 1 = transacción riesgosa
# 0 = transacción normal
df_ml = df_ml.withColumn(
    "label",
    F.when(
        (F.col("Monto_Total") > 7000) &
        (F.col("Hora_num") < 6),
        1
    ).otherwise(0)
)

# Distribución de la variable objetivo
df_ml.groupBy("label").count().orderBy("label").show()

+-----+-----+
|label|count|
+-----+-----+
|    0|  127|
|    1|    5|
+-----+-----+



### 1.3 Transformación de variables categóricas

Las variables sucursal y Producto son categóricas, por lo que deben convertirse a valores numéricos antes de ser utilizadas por el algoritmo de Machine Learning.

Para esto se utilizará StringIndexer, herramienta de MLlib que asigna un índice numérico a cada categoría.

Posteriormente, las variables numéricas y categóricas transformadas serán combinadas mediante VectorAssembler en una única columna denominada features.

In [ ]:
# StringIndexer

# Categorias de Sucursal a indices numéricos
indexer_sucursal = StringIndexer(
    inputCol="Sucursal",
    outputCol="Sucursal_index",
    handleInvalid="keep"
)

# categorias de Producto a ndices numéricos
indexer_producto = StringIndexer(
    inputCol="Producto",
    outputCol="Producto_index",
    handleInvalid="keep"
)

### 1.4 Construcción de la variable features

Una vez transformadas las variables categóricas, se utilizará VectorAssembler para reunir todas las variables predictoras en una sola columna llamada features.

Esta es la estructura requerida por los algoritmos de MLlib para entrenar el modelo.

In [ ]:
# Vector Assembler

# Se definen las variables que serán utilizadas como predictores
variables = [
    "Sucursal_index",
    "Producto_index",
    "Cantidad",
    "Precio_Unitario",
    "Monto_Total",
    "Hora_num"
]

# Unión de las variables en una sola columna de características
assembler = VectorAssembler(
    inputCols=variables,
    outputCol="features"
)

In [ ]:
# Pipeline de preprocesamiento

# Construcción de pipeline con las etapas de transformación
pipeline = Pipeline(
    stages=[
        indexer_sucursal,
        indexer_producto,
        assembler
    ]
)

# Ajuste del pipeline sobre el dataset
pipeline_model = pipeline.fit(df_ml)

# se aplicantransformaciones
df_preparado = pipeline_model.transform(df_ml)

In [ ]:
# Validación de dataset final

# Revisar las columnas principales para el modelo
df_preparado.select(
    "Sucursal",
    "Producto",
    "Cantidad",
    "Monto_Total",
    "Hora_num",
    "features",
    "label"
).show(10, truncate=False)

# Revisar nuevamente el esquema
df_preparado.printSchema()

+----------+--------+--------+-----------+--------+----------------------------------+-----+
|Sucursal  |Producto|Cantidad|Monto_Total|Hora_num|features                          |label|
+----------+--------+--------+-----------+--------+----------------------------------+-----+
|Sucursal A|Pan     |3       |2969.34    |17      |[3.0,2.0,3.0,989.78,2969.34,17.0] |0    |
|Sucursal A|Aceite  |4       |2254.28    |14      |[3.0,1.0,4.0,563.57,2254.28,14.0] |0    |
|Sucursal B|Galletas|4       |4722.0     |2       |[1.0,0.0,4.0,1180.5,4722.0,2.0]   |0    |
|Sucursal D|Pan     |3       |6584.97    |18      |[0.0,2.0,3.0,2194.99,6584.97,18.0]|0    |
|Sucursal B|Atún    |3       |7411.32    |23      |[1.0,5.0,3.0,2470.44,7411.32,23.0]|0    |
|Sucursal C|Leche   |3       |3631.62    |19      |[2.0,6.0,3.0,1210.54,3631.62,19.0]|0    |
|Sucursal B|Aceite  |2       |1653.6     |13      |[1.0,1.0,2.0,826.8,1653.6,13.0]   |0    |
|Sucursal C|Pan     |2       |4652.54    |2       |[2.0,2.0,2.0,2326.2

## 2. Entrenamiento del modelo de clasificación

Para predecir si una transacción es riesgosa se utilizará RandomForestClassifier de Apache Spark MLlib.

Random Forest fue seleccionado porque combina múltiples árboles de decisión y permite identificar relaciones no lineales entre las variables predictoras y la variable objetivo. Además, es adecuado para un problema de clasificación binaria como el planteado.

Antes de entrenar el modelo, se dividirá el dataset en un conjunto de entrenamiento y otro de prueba utilizando randomSplit().

###2.1 División del dataset

Se divide en 80% para entrenamiento y 20% para prueba

In [ ]:
#Selección solamente las columnas necesarias para el modelo
df_modelo = df_preparado.select(
    "features",
    "label"
)

# Division de los datos:
# 80% para entrenamiento
# 20% para prueba
train, test = df_modelo.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Cantidad de registros de cada conjunto
print("Registros de entrenamiento:", train.count())
print("Registros de prueba:", test.count())

Registros de entrenamiento: 102
Registros de prueba: 30


In [ ]:
#Revisión de la variable objetivo

print("Distribución en entrenamiento:")
train.groupBy("label").count().orderBy("label").show()

print("Distribución en prueba:")
test.groupBy("label").count().orderBy("label").show()

Distribución en entrenamiento:
+-----+-----+
|label|count|
+-----+-----+
|    0|   98|
|    1|    4|
+-----+-----+

Distribución en prueba:
+-----+-----+
|label|count|
+-----+-----+
|    0|   29|
|    1|    1|
+-----+-----+



### 2.2 Configuración y entrenamiento del Random Forest:

Se utilizará RandomForestClassifier utilizando la columna features como entrada y label como variable objetivo.Se configurará un número moderado de árboles debido al tamaño reducido del dataset.

In [ ]:
#Creación del modelo

# Creación d el clasificador Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    seed=42
)

# Entrenamiento del modelo utilizando los datos de entrenamiento
modelo_rf = rf.fit(train)

print("Modelo Random Forest entrenado correctamente")

Modelo Random Forest entrenado correctamente


### 2.3 Generación de predicciones

Una vez entrenado el modelo, se utilizará el conjunto de prueba para generar las predicciones.

Spark genera tres columnas relevantes:

- label: valor real de la transacción.
- prediction: clase predicha por el modelo.
- probability: probabilidad estimada para cada clase.

In [ ]:
# Generación de predicciones

# Se generan predicciones sobre el conjunto de prueba
predicciones = modelo_rf.transform(test)

# Mostrar las columnas solicitadas
predicciones.select(
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)

+-----+----------+-----------+
|label|prediction|probability|
+-----+----------+-----------+
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[0.9,0.1]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[0.92,0.08]|
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[0.7,0.3]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[0.96,0.04]|
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[0.94,0.06]|
|0    |0.0       |[0.94,0.06]|
|0    |0.0       |[0.82,0.18]|
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
|0    |0.0       |[1.0,0.0]  |
+-----+----------+-----------+
only showing top 20 rows


In [ ]:
#Resumen de predicciones

# Contamos cuantas predicciones corresponden a cada clase
predicciones.groupBy(
    "label",
    "prediction"
).count().orderBy(
    "label",
    "prediction"
).show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|   29|
|    1|       0.0|    1|
+-----+----------+-----+



## 3. Evaluar el modelo y justificar decisiones técnicas

Para evaluar el desempeño del modelo se utilizarán tres métricas de clasificación:

- Accuracy: proporción de predicciones correctas sobre el total de predicciones.
- F1-score:combina precision y recall, siendo especialmente útil cuando existe desbalance entre las clases.
- Area Under ROC (AUC-ROC): mide la capacidad del modelo para distinguir entre transacciones normales y riesgosas.



### 3.1 Evaluación de Acurracy , F1-Score y AUC-ROC

In [ ]:
# Evaluación Acurracy

#Evaluador para calcular accuracy
evaluador_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Calculo de accuracy
accuracy = evaluador_accuracy.evaluate(predicciones)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9667


In [ ]:
# Evaluación F1-Score

# Evaluador para calcular F1-score
evaluador_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

# F1-score
f1 = evaluador_f1.evaluate(predicciones)

print(f"F1-score: {f1:.4f}")

F1-score: 0.9503


In [ ]:
# Evaluación ROC -AUC

# Evaluador binario
evaluador_roc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# AUC-ROC
auc = evaluador_roc.evaluate(predicciones)

print(f"Area Under ROC (AUC): {auc:.4f}")

Area Under ROC (AUC): 0.9310


### 3.2 Interpretación y justificación

El modelo Random Forest fue seleccionado debido a su capacidad para identificar relaciones no lineales entre las variables de las transacciones y la clasificación de riesgo. Además, permite trabajar con múltiples variables predictoras y obtener información sobre su importancia.

Para evaluar el modelo se utilizaron Accuracy, F1-score y AUC-ROC. El Accuracy permite conocer el porcentaje general de predicciones correctas, mientras que F1-score permite evaluar de mejor manera el desempeño cuando existe desbalance entre las clases. AUC-ROC permite medir la capacidad del modelo para diferenciar entre transacciones normales y riesgosas.

Los resultados deben interpretarse considerando que la clase riesgosa es minoritaria. Por este motivo, un Accuracy elevado por sí solo no garantiza una buena detección de transacciones riesgosas.

Como mejora futura, se podría aplicar validación cruzada y ajuste de hiperparámetros, además de incorporar nuevas variables históricas de comportamiento de las ventas. También sería conveniente evaluar técnicas para abordar el desbalance de clases y mejorar la detección de transacciones riesgosas.